# Exploratory Data Analysis
## Machine Health Monitoring & Fault Diagnosis System

This notebook walks through the AI4I 2020 Predictive Maintenance dataset:
1. Load & overview  
2. Missing values & distributions  
3. Correlation analysis  
4. Fault analysis by operating condition  
5. Feature engineering preview  
6. Health score visualisation

**Run order:** execute cells top-to-bottom. Ensure you have run `python main.py --steps 1 2 3 4` first.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

PROC = Path('../data/processed')
RAW  = Path('../data/raw')

COLORS = {
    'Good':     '#27ae60',
    'Warning':  '#f39c12',
    'Degraded': '#e67e22',
    'Critical': '#e74c3c',
    'primary':  '#2c3e50',
    'accent':   '#2980b9',
}
print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
# Load whichever stage is available (prefer scored, fall back to raw)
def load_best_available():
    for fname in ['ai4i_health_scored.csv', 'ai4i_features.csv', 'ai4i_clean.csv']:
        p = PROC / fname
        if p.exists():
            df = pd.read_csv(p)
            print(f'Loaded: {fname}  ({df.shape[0]:,} rows × {df.shape[1]} cols)')
            return df, fname
    # Fall back to raw
    raw = RAW / 'ai4i2020.csv'
    if raw.exists():
        df = pd.read_csv(raw)
        print(f'Loaded raw: ai4i2020.csv  ({df.shape[0]:,} rows × {df.shape[1]} cols)')
        return df, 'raw'
    raise FileNotFoundError('No data found. Run: python main.py --steps 1 2')

df, source = load_best_available()
df.head()

In [ ]:
print('=== Dataset Info ===')
print(df.dtypes)
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

## 2. Missing Values & Distributions

In [ ]:
# Missing value heatmap
missing = df.isnull().sum()
missing = missing[missing > 0]

if missing.empty:
    print('No missing values ✓')
else:
    fig, ax = plt.subplots(figsize=(10, 3))
    missing.sort_values().plot.barh(ax=ax, color=COLORS['Critical'])
    ax.set_title('Missing Value Count per Column', fontweight='bold')
    ax.set_xlabel('# Missing')
    plt.tight_layout()
    plt.show()
    print(missing)

In [ ]:
# Distribution of numeric sensor columns
sensor_cols = [c for c in [
    'air_temp_K', 'process_temp_K',
    'rotational_speed_rpm', 'torque_Nm', 'tool_wear_min'
] if c in df.columns]

if sensor_cols:
    n = len(sensor_cols)
    fig, axes = plt.subplots(2, n, figsize=(3.5 * n, 6), gridspec_kw={'hspace': 0.4})

    for i, col in enumerate(sensor_cols):
        # Histogram
        axes[0, i].hist(df[col].dropna(), bins=40,
                        color=COLORS['accent'], edgecolor='white', linewidth=0.4)
        axes[0, i].set_title(col.replace('_', '\n'), fontsize=9, fontweight='bold')
        axes[0, i].set_ylabel('Count' if i == 0 else '')

        # Box plot
        axes[1, i].boxplot(df[col].dropna(), patch_artist=True,
                           boxprops=dict(facecolor=COLORS['accent'], alpha=0.5),
                           medianprops=dict(color=COLORS['primary'], linewidth=2))
        axes[1, i].set_xticks([])

    fig.suptitle('Sensor Column Distributions', fontsize=13, fontweight='bold', y=1.01)
    plt.show()

print(df[sensor_cols].describe().T.round(3))

## 3. Fault Analysis

In [ ]:
fault_col = 'fault_type' if 'fault_type' in df.columns else 'machine_failure'

if fault_col in df.columns:
    counts = df[fault_col].value_counts()
    pcts   = (counts / len(df) * 100).round(2)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Bar
    bar_colors = [COLORS['Good'] if v == 'Normal' else COLORS['Critical']
                  for v in counts.index]
    bars = axes[0].bar(counts.index, counts.values, color=bar_colors, edgecolor='white')
    axes[0].bar_label(bars, labels=[f'{v:,}\n({p}%)' for v, p in zip(counts.values, pcts)],
                      padding=3, fontsize=9)
    axes[0].set_title('Fault Type Count', fontweight='bold')
    axes[0].set_xlabel('Fault Type')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=20)

    # Pie
    pie_cols = [COLORS['Good'] if v == 'Normal' else COLORS['Critical']
                for v in counts.index]
    axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%',
                colors=pie_cols, startangle=140)
    axes[1].set_title('Fault Type Distribution', fontweight='bold')

    plt.suptitle('Fault Analysis', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

print(f'Overall failure rate: {(df["machine_failure"].mean() if "machine_failure" in df.columns else 0):.2%}')

In [ ]:
# Sensor values split by fault type — violin plots
if fault_col in df.columns and sensor_cols:
    n = len(sensor_cols)
    fig, axes = plt.subplots(1, n, figsize=(3.8 * n, 4.5), sharey=False)

    palette = {v: (COLORS['Good'] if v == 'Normal' else COLORS['Critical'])
               for v in df[fault_col].unique()}

    for i, col in enumerate(sensor_cols):
        ax = axes[i] if n > 1 else axes
        groups = [df.loc[df[fault_col] == g, col].dropna().values
                  for g in df[fault_col].unique()]
        parts = ax.violinplot(groups, showmedians=True)
        for pc, color in zip(parts['bodies'], palette.values()):
            pc.set_facecolor(color)
            pc.set_alpha(0.6)
        ax.set_xticks(range(1, len(palette) + 1))
        ax.set_xticklabels(list(palette.keys()), rotation=20, fontsize=8)
        ax.set_title(col.replace('_', '\n'), fontsize=9, fontweight='bold')
        ax.set_ylabel('Value' if i == 0 else '')

    plt.suptitle('Sensor Distributions by Fault Type', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 4. Correlation Analysis

In [ ]:
num_cols = df.select_dtypes('number').columns.tolist()
num_cols = [c for c in num_cols if 'uid' not in c.lower()
            and not c.isupper()][:12]   # skip uid and dummy flags

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))   # show lower triangle only
sns.heatmap(
    corr,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, ax=ax,
    annot_kws={'size': 8},
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix (lower triangle)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Top 10 strongest correlations
pairs = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
)
pairs.columns = ['feature_1', 'feature_2', 'correlation']
pairs['abs_corr'] = pairs['correlation'].abs()
print('Top 10 correlated pairs:')
print(pairs.nlargest(10, 'abs_corr')[['feature_1','feature_2','correlation']].to_string(index=False))

## 5. Health Score Analysis

In [ ]:
if 'health_score' not in df.columns:
    print('health_score not found — run python main.py --steps 4 first.')
else:
    hs = df['health_score']
    print(f'Health Score Statistics')
    print(f'  Mean   : {hs.mean():.2f}')
    print(f'  Median : {hs.median():.2f}')
    print(f'  Std    : {hs.std():.2f}')
    print(f'  Min    : {hs.min():.2f}')
    print(f'  Max    : {hs.max():.2f}')

    if 'health_status' in df.columns:
        print('\nStatus breakdown:')
        print(df['health_status'].value_counts())

In [ ]:
if 'health_score' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Histogram with threshold lines
    axes[0].hist(df['health_score'], bins=40,
                 color=COLORS['accent'], edgecolor='white', linewidth=0.4)
    for thr, lbl in [(80, 'Good'), (60, 'Warning'), (40, 'Degraded')]:
        axes[0].axvline(thr, color=COLORS[lbl], linestyle='--',
                        linewidth=1.4, label=f'{lbl} ({thr})')
    axes[0].set_xlabel('Health Score (0–100)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Health Score Distribution', fontweight='bold')
    axes[0].legend(fontsize=9)

    # Trend over observations
    sample = df['health_score'].head(2000)
    axes[1].scatter(sample.index, sample.values,
                    c=sample.values, cmap='RdYlGn', s=4, alpha=0.5,
                    vmin=0, vmax=100)
    # Rolling mean
    rolling = sample.rolling(100, min_periods=1).mean()
    axes[1].plot(rolling.index, rolling.values,
                 color=COLORS['primary'], linewidth=1.5, label='Rolling mean')
    for thr, lbl in [(80, 'Good'), (60, 'Warning'), (40, 'Degraded')]:
        axes[1].axhline(thr, color=COLORS[lbl], linestyle='--', linewidth=0.9)
    axes[1].set_xlabel('Observation Index')
    axes[1].set_ylabel('Health Score')
    axes[1].set_title('Health Score Trend', fontweight='bold')
    axes[1].legend()

    plt.suptitle('Asset Health Score Overview', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 6. Operating Condition Heatmap

In [ ]:
# Failure rate vs Rotational Speed × Torque (2D heatmap)
if all(c in df.columns for c in ['rotational_speed_rpm', 'torque_Nm', 'machine_failure']):
    df['speed_bin']  = pd.cut(df['rotational_speed_rpm'], bins=10)
    df['torque_bin'] = pd.cut(df['torque_Nm'], bins=10)

    pivot = df.pivot_table(
        values='machine_failure',
        index='torque_bin',
        columns='speed_bin',
        aggfunc='mean'
    ) * 100

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(
        pivot,
        cmap='YlOrRd',
        annot=True, fmt='.1f',
        linewidths=0.3,
        ax=ax,
        cbar_kws={'label': 'Failure Rate (%)'}
    )
    ax.set_title('Failure Rate (%) by Speed × Torque Operating Region',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Rotational Speed (rpm)')
    ax.set_ylabel('Torque (Nm)')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('Speed/Torque/Failure columns not found — skipping heatmap.')

## 7. Tool Wear Progression

In [ ]:
if 'tool_wear_min' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    failure_col = 'machine_failure' if 'machine_failure' in df.columns else None

    # Tool wear vs health score (scatter)
    if 'health_score' in df.columns:
        scatter_c = df.get('health_score', None)
        sc = axes[0].scatter(
            df['tool_wear_min'], df['health_score'],
            c=scatter_c, cmap='RdYlGn', s=5, alpha=0.4,
            vmin=0, vmax=100
        )
        plt.colorbar(sc, ax=axes[0], label='Health Score')
        axes[0].set_xlabel('Tool Wear (min)')
        axes[0].set_ylabel('Health Score')
        axes[0].set_title('Tool Wear vs Health Score', fontweight='bold')

    # Tool wear histogram coloured by failure
    if failure_col:
        normal = df.loc[df[failure_col] == 0, 'tool_wear_min']
        failed = df.loc[df[failure_col] == 1, 'tool_wear_min']
        axes[1].hist(normal, bins=30, alpha=0.7, color=COLORS['Good'],   label='Normal')
        axes[1].hist(failed, bins=30, alpha=0.7, color=COLORS['Critical'], label='Failure')
        axes[1].set_xlabel('Tool Wear (min)')
        axes[1].set_ylabel('Count')
        axes[1].set_title('Tool Wear Distribution by Outcome', fontweight='bold')
        axes[1].legend()

    plt.suptitle('Tool Wear Analysis', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Correlation with failure
    if failure_col:
        r, p = stats.pointbiserialr(df[failure_col], df['tool_wear_min'].fillna(0))
        print(f'Point-biserial correlation (tool_wear vs failure): r={r:.3f}, p={p:.2e}')

## 8. Key Takeaways

Summarise findings from this EDA before proceeding to modelling:

| Finding | Detail |
|---|---|
| Dataset size | ~10,000 observations, 14 original columns |
| Failure rate | ~3.4% — class imbalance; use `class_weight='balanced'` |
| Top correlated pair | `process_temp_K` ↔ `air_temp_K` (high correlation) |
| Strongest failure predictor | `tool_wear_min` + `torque_Nm` |
| Most common fault | HDF (Heat Dissipation Failure) |

> Edit this table with your actual results after running the cells above.

In [ ]:
# Export a summary stats CSV for the report
if sensor_cols:
    summary = df[sensor_cols].describe().T
    out = PROC / 'eda_summary_stats.csv'
    summary.to_csv(out)
    print(f'Summary stats saved → {out}')
    print(summary.round(3))